# AB-200 Questions: Q1–10 — Python Syntax and Fundamentals

Part of a series — each notebook in this set covers 10 questions from the AB-200 list (Q1–10, Q11–20, Q21–30, ...).

For each question we follow the same drill, in this order:

1. **Original question** — quoted as written, with its hint, so the notebook stands on its own.
2. **Restate the problem** — in plain language, so we're sure we're solving the right thing.
3. **Think algorithmically** — write the steps as an ordinary person would describe them, with no loops/ifs/syntax. This is the thinking a whiteboard interview actually tests.
4. **Brute force** — the first correct idea that comes to mind, even if it's wasteful.
5. **Optimal** — built *on top of* the brute force by asking "what work are we repeating, and can we avoid it?"
6. **Tests** — normal case, edge cases (empty / single / negative / duplicate), and asserts that actually run.
7. **Interview traps** — the specific mistakes that make candidates lose points on this exact question.

---
## Q1. Swap Variables Without Temp

> **Swap Variables Without Temp:** Given two variables `x` and `y`, swap their values without using a third variable.
> _Hint:_ Python's tuple unpacking allows swapping values in one line.

**Restate:** We have two variables, `x` and `y`. We want `x` to end up holding `y`'s original value, and `y` to end up holding `x`'s original value — without introducing a third named variable to hold one of them while we overwrite the other.

**Algorithmic thinking (no syntax):**
1. Remember both original values at once, as a pair.
2. Assign the pair back in reversed order: first slot gets what was in the second slot, second slot gets what was in the first slot.

Notice this sidesteps the "temp variable" problem entirely — we never overwrite a value before we've captured *both* of them. The classic swap-with-temp approach is really solving a different, more primitive constraint: *you can only read/write one box at a time*. Once you allow "read both, then write both" as a single atomic step, the temp variable was never needed.

In [ ]:
# Brute force — the "temp variable" way.
# This is what almost every beginner writes first. Nothing wrong with it,
# it's just doing one more assignment than it strictly needs to.

def swap_brute(x, y):
    temp = x
    x = y
    y = temp
    return x, y

print(swap_brute(1, 2))   # (2, 1)

**Optimal:** Python lets you build the pair explicitly with a tuple, and unpack it in one statement. The right-hand side `(y, x)` is fully evaluated — both values are read — *before* any assignment on the left happens. That's exactly the "read both, then write both" step from our pseudocode, made literal.

There's also an arithmetic trick for numbers only (`x = x + y; y = x - y; x = x - y`) — mention it if asked, but call out that it overflows/breaks for non-numeric types and is genuinely harder to read. It's a "cute" answer, not a better one.

In [ ]:
# Optimal — tuple unpacking. Same O(1) time as the brute force,
# but one line and no extra name in scope.

def swap_optimal(x, y):
    x, y = y, x
    return x, y

print(swap_optimal(1, 2))   # (2, 1)

In [ ]:
# Tests

assert swap_optimal(1, 2) == (2, 1)
assert swap_optimal("a", "b") == ("b", "a")
assert swap_optimal(x=5, y=5) == (5, 5)          # equal values: still correct, just a no-op
assert swap_optimal([1], [2]) == ([2], [1])       # works for any object, not just numbers
assert swap_brute(1, 2) == swap_optimal(1, 2)     # both approaches agree

print("All Q1 tests passed.")

**Interview traps:**
- If asked to do it "without a third variable" and you reach for the arithmetic trick, be ready to explain it breaks on strings/lists/floats-with-precision-loss — interviewers often push on this to see if you understand *why* it works, not just that it does.
- In real code, swapping two elements *inside a list* (`lst[i], lst[j] = lst[j], lst[i]`) uses the same trick — this is worth mentioning proactively since it's the actual use case that shows up inside sorting algorithms later in the set.
- This "problem" only exists because some languages (C, Java before records) don't have tuple unpacking. Saying that shows you understand it's a language-feature question, not an algorithms question.

---
## Q2. Reverse a String

> **Reverse a String:** Given a string, return the string reversed.
> _Hint:_ Use Python slicing like `s[::-1]` or the built-in `reversed()` function to reverse the sequence.

**Restate:** Given a sequence of characters, produce a new sequence containing the same characters in the opposite order.

**Algorithmic thinking (no syntax):**
1. Start a new, empty sequence.
2. Walk the original sequence from its *last* position to its *first*.
3. At each step, place the character you're looking at onto the end of the new sequence.
4. When you've walked past the first character, the new sequence is the answer.

An equivalent way to think about it, which turns out to matter for the optimal version: pair up position `0` with the last position, `1` with second-to-last, and so on, and swap each pair. You don't need a *new* sequence at all if you're allowed to swap in place — you just need to know when the two pointers meet in the middle.

In [ ]:
# Brute force — literally walk backwards and build a new string.
# O(n) time, O(n) extra space for the result (unavoidable for strings,
# more on that below). The wasteful part is `result = char + result`:
# strings are immutable, so *every* concatenation copies the whole
# string built so far. That makes this O(n^2), not O(n).

def reverse_brute(s):
    result = ""
    for i in range(len(s) - 1, -1, -1):
        result = result + s[i]
    return result

print(reverse_brute("hello"))   # "olleh"

**Optimal:** Two ways worth knowing, for different reasons.

- **Slicing** `s[::-1]` — asks the string for "every element, stepping backwards by 1". It's C-implemented under the hood, so it's the fastest option and the one you'd actually write in production.
- **`reversed()`** — returns an iterator that yields characters back-to-front *without* building the reversed string until you ask for one (e.g. via `"".join(...)`). This matters when you only need to *scan* the string backwards (like Q3's palindrome check) and don't need the reversed copy materialized at all.

Both are O(n) time. `s[::-1]` is O(n) space immediately; `reversed()` is O(1) extra space until you materialize it.

In [ ]:
# Optimal

def reverse_slice(s):
    return s[::-1]

def reverse_iterator(s):
    return "".join(reversed(s))

print(reverse_slice("hello"))      # "olleh"
print(reverse_iterator("hello"))   # "olleh"

In [ ]:
# Tests

assert reverse_slice("hello") == "olleh"
assert reverse_slice("") == ""                # edge: empty string
assert reverse_slice("a") == "a"              # edge: single character
assert reverse_slice("ab") == "ba"            # edge: even length
assert reverse_slice("madam") == "madam"      # palindrome: reversed equals original
assert reverse_slice("a b") == "b a"          # spaces are characters too

for fn in (reverse_brute, reverse_slice, reverse_iterator):
    assert fn("interview") == "weivretni", fn.__name__

print("All Q2 tests passed.")

In [ ]:
# The trap, demonstrated: reversing a string with a "combined" character
# (a skin-tone modifier attached to an emoji) reverses the *code points*,
# which can visually break the emoji apart from its modifier.

combined = "a👍🏽b"          # 👍🏽 is actually TWO code points: 👍 + a skin-tone modifier
print(list(combined))       # see how many "characters" Python thinks this is
print(reverse_slice(combined))

**Interview traps:**
- **"Don't use slicing"** is a common follow-up constraint — it's really asking you to demonstrate the two-pointer swap-in-place idea from the pseudocode. Know it cold (see the code cell below).
- **Strings are immutable.** You cannot do `s[i], s[j] = s[j], s[i]` on a `str` the way you can on a `list` — that line raises `TypeError`. The two-pointer version has to convert to a `list` first, reverse in place, then `"".join()` back. If you forget this, you'll write code that doesn't compile and look flustered live.
- **Concatenation in a loop is the silent O(n²) trap** — the brute force above builds the trap on purpose so you can point at it. `str1 + str2` always copies both. Building a string by repeated `+=` inside a loop is the single most common "I wrote correct-looking code that's secretly quadratic" mistake.
- **Unicode/grapheme clusters** (shown above) — reversing by code point can separate an emoji from its modifier, or break apart accented characters stored as base+combining-mark pairs. Mentioning this unprompted is a strong signal in an interview; it almost never needs to be *solved*, just *named*.

In [ ]:
# The "no slicing, no reversed()" follow-up: two pointers, in place,
# on a list of characters (since str itself can't be mutated).

def reverse_two_pointer(s):
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

assert reverse_two_pointer("hello") == "olleh"
assert reverse_two_pointer("") == ""
assert reverse_two_pointer("a") == "a"
print(reverse_two_pointer("interview"))   # "weivretni"

---
## Q3. Palindrome Check

> **Palindrome Check:** Check if a given string reads the same forwards and backwards (case-sensitive, no spaces removed).
> _Hint:_ Compare the string to its reverse (e.g., `s == s[::-1]`).

**Restate:** Given a string, decide whether reading it forwards produces the exact same sequence of characters as reading it backwards. (This variant is case-sensitive and does **not** strip spaces — `"a a"` reversed is `"a a"`, still a palindrome; `"Aa"` is **not**, because `'A' != 'a'`.)

**Algorithmic thinking (no syntax):**
1. Point at the first character and the last character.
2. Compare them. If they differ, it's not a palindrome — stop immediately, no need to check the rest.
3. If they match, move both pointers one step toward the middle (first pointer forward, last pointer backward).
4. Repeat until the pointers meet or cross. If you never found a mismatch, it's a palindrome.

Notice step 2's early exit: the moment you find *one* mismatch, you know the answer. A solution that builds the full reversed string first and then compares can't stop early — it does all the work regardless of where the mismatch is.

In [ ]:
# Brute force — build the full reverse, then compare.
# O(n) time, O(n) extra space. Correct, but always does the full
# reversal even if the very first pair of characters already disagrees.

def is_palindrome_brute(s):
    return s == s[::-1]

print(is_palindrome_brute("madam"))   # True
print(is_palindrome_brute("hello"))   # False

**Optimal:** The two-pointer walk from the pseudocode, coded directly. Worst case (an actual palindrome, or a mismatch only at the very center) is still O(n) time — same as the brute force asymptotically. The real win is **O(1) extra space** (no reversed copy ever built) and **early exit** on the common case of an obvious non-palindrome, which in practice touches far fewer than `n` characters.

This is the answer to give when asked to do it "in O(1) space" or "without slicing."

In [ ]:
# Optimal — two pointers, O(1) extra space, early exit.

def is_palindrome_optimal(s):
    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            return False
        left += 1
        right -= 1
    return True

print(is_palindrome_optimal("madam"))   # True
print(is_palindrome_optimal("hello"))   # False

In [ ]:
# Tests

assert is_palindrome_optimal("madam") is True
assert is_palindrome_optimal("hello") is False
assert is_palindrome_optimal("") is True           # edge: empty string — vacuously true
assert is_palindrome_optimal("a") is True          # edge: single character
assert is_palindrome_optimal("aa") is True          # edge: two identical characters
assert is_palindrome_optimal("ab") is False         # edge: two different characters
assert is_palindrome_optimal("a a") is True         # spaces are NOT stripped, by spec
assert is_palindrome_optimal("Aa") is False         # case-sensitive, by spec

for fn in (is_palindrome_brute, is_palindrome_optimal):
    assert fn("racecar") is True, fn.__name__
    assert fn("Racecar") is False, fn.__name__   # case sensitivity re-checked on both impls

print("All Q3 tests passed.")

**Interview traps:**
- **Clarify the spec before coding.** "Palindrome" in casual usage ignores case and spaces/punctuation ("A man, a plan, a canal: Panama"). This question explicitly says case-sensitive, spaces kept — if you silently normalize the string anyway (`.lower()`, strip spaces), you've solved a different, easier problem than the one asked. Say your assumption out loud either way.
- **Empty string and single character are both trivially palindromes** — the loop condition `left < right` handles both automatically (loop body never runs), but you should be able to say *why* without running it: with 0 or 1 characters there's no pair that could possibly mismatch.
- **`s == s[::-1]` is a perfectly fine answer** for a simple ask, but if the interviewer says "O(1) space" or "without creating a copy," reaching for slicing anyway is the tell that you don't know what's happening under the hood.
- If the real spec *does* want case-insensitive / alphanumeric-only comparison (the classic LeetCode 125 version), normalize first (`s = s.lower()`, filter to `.isalnum()`) and *then* run the same two-pointer walk — don't conflate "cleaning the input" with "checking the palindrome property," they're separate steps.

---
## Q4. Factorial Calculation

> **Factorial Calculation:** Compute the factorial of a non-negative integer n (n! = n × (n-1) × ... × 1).
> _Hint:_ Use a loop or recursion; 0! is 1 as the base case.

**Restate:** Given a non-negative whole number `n`, compute the product of every whole number from `n` down to `1`. By definition, the product of "nothing" (`n = 0`) is `1`.

**Algorithmic thinking (no syntax):**
1. Start an accumulated product at `1`.
2. Starting from `n` and counting down to `1`, multiply the accumulated product by the current number, one number at a time.
3. When you reach `1`, the accumulated product is the answer.

Equivalently, phrased the way it's usually *defined* rather than *computed*: "the factorial of `n` is `n` times the factorial of `n - 1`, and the factorial of `0` is `1`." That definition is self-referential — it describes the answer for `n` in terms of the same problem on a smaller number, `n - 1` — which is exactly what recursion is for. Both phrasings compute the identical sequence of multiplications; they differ in *who* keeps track of the running product (you, with a variable, vs. the call stack, with pending multiplications).

In [ ]:
# "Brute force" here is the recursive, self-referential version.
# It's not inefficient the way earlier brute forces were — it's O(n)
# time just like the loop — but it costs O(n) *call-stack space*,
# and Python's default recursion limit (~1000) means it breaks for
# large n in a way the loop never does. That's the trade we're
# building on top of below.

def factorial_recursive(n):
    if n < 0:
        raise ValueError("factorial is undefined for negative numbers")
    if n == 0:                 # base case — stops the self-reference
        return 1
    return n * factorial_recursive(n - 1)

print(factorial_recursive(5))   # 120
print(factorial_recursive(0))   # 1

**Optimal:** The iterative version does the same multiplications with O(1) extra space — no call stack growth, so no recursion-limit ceiling. In real code, `math.factorial` is the actual right answer: it's implemented in C and used by the standard library itself, so there's no reason to hand-roll this outside of an interview.

In [ ]:
import math

# Optimal — iterative, O(n) time, O(1) extra space.

def factorial_iterative(n):
    if n < 0:
        raise ValueError("factorial is undefined for negative numbers")
    product = 1
    for i in range(2, n + 1):   # starting at 2: multiplying by 1 or 0 is a no-op
        product *= i
    return product

print(factorial_iterative(5))       # 120
print(math.factorial(5))            # 120 — the real-world answer

In [ ]:
# Tests

assert factorial_iterative(0) == 1          # edge: base case
assert factorial_iterative(1) == 1          # edge: smallest non-trivial input
assert factorial_iterative(5) == 120
assert factorial_iterative(10) == 3628800
assert factorial_iterative(20) == math.factorial(20)   # Python ints don't overflow

for n in range(0, 15):
    assert factorial_recursive(n) == factorial_iterative(n) == math.factorial(n)

try:
    factorial_iterative(-1)
    assert False, "expected ValueError for negative input"
except ValueError:
    pass   # edge: negative input correctly rejected

print("All Q4 tests passed.")

**Interview traps:**
- **Negative input.** "Non-negative integer n" in the prompt is doing real work — decide and state what happens for `n = -1` (raise, per above, is the defensible choice) rather than silently returning something. Forgetting this check entirely is the single most common miss on this question.
- **0! = 1, not 0 and not undefined.** If you don't special-case it (or if your loop range excludes it), a naive "multiply from n down to 1" reading can accidentally return the wrong thing for `n = 0` depending on how the loop is written. Trace `n = 0` through your own code before running it.
- **"Unlike other languages, this can't overflow"** is worth saying out loud: Python `int` is arbitrary-precision, so `factorial(1000)` just works (slowly), whereas the same code in C/Java/Go silently wraps or needs a bignum library. It signals you know Python's number model, not just the algorithm.
- **Recursion depth.** `factorial_recursive(2000)` will hit `RecursionError` — if you offer the recursive solution as your primary answer, be ready for "what happens at n = 100,000?" and pivot to the iterative version, don't defend recursion as if it's free.

---
## Q5. Fibonacci Sequence

> **Fibonacci Sequence:** Generate the first n numbers of the Fibonacci sequence (where Fib(0)=0, Fib(1)=1).
> _Hint:_ Use iteration or recursion; store previous two numbers to compute the next.

**Restate:** Produce the first `n` numbers of the sequence where each number is the sum of the two before it, starting `0, 1`. So `Fib(0) = 0`, `Fib(1) = 1`, `Fib(2) = 1`, `Fib(3) = 2`, `Fib(4) = 3`, `Fib(5) = 5`, ...

**Algorithmic thinking (no syntax):**
1. Keep two running numbers in mind: the "previous" one and the "one before that."
2. The next number in the sequence is always just those two added together.
3. After producing a number, slide your attention forward: what used to be "previous" becomes "the one before that," and the number you just produced becomes the new "previous."
4. Repeat until you've produced `n` numbers.

This is different in character from factorial. Factorial's recursive definition only ever refers to *one* smaller subproblem (`n-1`). Fibonacci's natural definition — "`Fib(k)` is `Fib(k-1) + Fib(k-2)`" — refers to *two*. That branching is exactly what makes the naive recursive version dangerous, as we're about to see.

In [ ]:
# Brute force — naive recursion, transcribed straight from the definition.
# Looks innocent. It is exponential: fib(k) calls fib(k-1) and fib(k-2),
# each of which call two more, etc. — a binary tree of calls with
# roughly 2^k nodes. fib(30) already makes over a million calls.

def fib_naive(k):
    if k < 2:
        return k              # base cases: fib(0) = 0, fib(1) = 1
    return fib_naive(k - 1) + fib_naive(k - 2)

def fibonacci_sequence_brute(n):
    return [fib_naive(k) for k in range(n)]

print(fibonacci_sequence_brute(10))   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

In [ ]:
# Let's actually feel the exponential blowup, not just take it on faith.

import time

for k in (20, 25, 30, 32):
    start = time.time()
    fib_naive(k)
    elapsed = time.time() - start
    print(f"fib_naive({k}) took {elapsed:.3f}s")

Each +5 to `k` roughly multiplies the time by ~11 (≈ φ⁵, since the call count grows like φᵏ where φ ≈ 1.618). Doubling `n` doesn't double the work — this is what "exponential" actually feels like on a keyboard, not just in a Big-O table.

**Why it's exponential — the repeated-work diagnosis:** `fib_naive(5)` calls `fib_naive(3)` (via the `k-2` branch) *and* `fib_naive(4)`, which itself calls `fib_naive(3)` again (via its own `k-2` branch). `fib_naive(3)` gets computed from scratch twice, and this compounds at every level. The fix, in both directions, is to stop recomputing the same subproblem:

- **Memoize the recursion** — cache each `fib_naive(k)` result the first time it's computed, so the second call is a lookup instead of a recomputation.
- **Or flip to iteration** — walk forward from the base cases instead of branching backward from `n`, so each value is computed exactly once by construction, and no cache is even needed.

Both get you from exponential to linear. The iterative version is the one to actually offer as your "optimal" answer, since it also uses O(1) space instead of memoization's O(n).

In [ ]:
# Bridge step — memoized recursion. Same shape as fib_naive, but a
# cache dict means each k is computed once. O(n) time, O(n) space
# (both the cache and the call stack).

from functools import lru_cache

@lru_cache(maxsize=None)
def fib_memo(k):
    if k < 2:
        return k
    return fib_memo(k - 1) + fib_memo(k - 2)

start = time.time()
fib_memo(100)          # instant — would take longer than the age of the universe naively
print(fib_memo(100), f"({time.time() - start:.6f}s)")

In [ ]:
# Optimal — iterative, O(n) time, O(1) extra space, no recursion limit,
# no cache. This is the version to lead with.

def fibonacci_sequence_optimal(n):
    if n <= 0:
        return []
    sequence = []
    prev, curr = 0, 1
    for _ in range(n):
        sequence.append(prev)
        prev, curr = curr, prev + curr   # the "slide forward" step from the pseudocode
    return sequence

print(fibonacci_sequence_optimal(10))   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

In [ ]:
# Tests

assert fibonacci_sequence_optimal(0) == []                    # edge: n = 0, no numbers requested
assert fibonacci_sequence_optimal(1) == [0]                    # edge: just the first number
assert fibonacci_sequence_optimal(2) == [0, 1]                 # edge: exactly the two seeds
assert fibonacci_sequence_optimal(10) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
assert fibonacci_sequence_optimal(-3) == []                    # edge: negative n treated as "none"

for n in range(0, 20):
    assert fibonacci_sequence_optimal(n) == fibonacci_sequence_brute(n)

assert [fib_memo(k) for k in range(15)] == fibonacci_sequence_optimal(15)

print("All Q5 tests passed.")

**Interview traps:**
- **"Compute Fib(n)" vs. "generate the first n Fibonacci numbers" are different questions** — one wants a single number (index into the sequence), the other wants a list. Read the prompt carefully; this question asks for the latter. Confusing the two mid-interview is common and easy to catch yourself doing if you restate the problem first (step 1 of our process).
- **Off-by-one on indexing.** Is `Fib(1)` the first `1` or the second `0`? Different sources 0-index vs 1-index the sequence differently. State your convention (`Fib(0) = 0, Fib(1) = 1`, as given) and make sure your loop's first and last iteration actually match it — this is the single most common silent bug on this question.
- **Never lead with naive recursion as your "solution"** unless explicitly asked to start brute-force. If you write it first, immediately flag that it's exponential and say what you'd do instead — that's a stronger signal than jumping straight to iteration with no reasoning shown.
- **`lru_cache` on a function with default/mutable arguments is dangerous in general** (not an issue here since `fib_memo` takes a plain int), but it's worth knowing: `lru_cache` requires arguments to be hashable, so it silently fails (`TypeError`) if you ever memoize a function that takes a `list` or `dict`.
- **Golden ratio closed form exists** (`Fib(n) ≈ φⁿ/√5`) and is O(1) per term — mentioning it shows depth, but floating-point rounding makes it unreliable for large `n`, so don't offer it as your primary answer.

---
## Q6. Prime Number Test

> **Prime Number Test:** Determine if a given number is prime (only divisible by 1 and itself).
> _Hint:_ Check divisibility from 2 up to sqrt(n); if no divisors found, it's prime.

**Restate:** Given a whole number `n`, decide whether its only divisors are `1` and itself. By convention `n = 0` and `n = 1` are **not** prime, and negative numbers aren't prime either (primality is only defined for integers ≥ 2).

**Algorithmic thinking (no syntax):**
1. If the number is less than 2, it's automatically not prime — stop.
2. Otherwise, try dividing it by every whole number from 2 up to (but not including) itself.
3. If any of those divides evenly (no remainder), it's not prime — stop.
4. If none of them divide evenly, it's prime.

The key insight that gets you from brute force to optimal: divisors come in **pairs that multiply to `n`**. If `n = a × b`, then one of `a` or `b` must be ≤ √n (they can't both be bigger, or their product would exceed `n`). So you only ever need to search up to √n — if no divisor shows up by then, none exists beyond it either.

In [ ]:
# Brute force — check every candidate divisor from 2 up to n-1.
# O(n) time.

def is_prime_brute(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

print(is_prime_brute(17))   # True
print(is_prime_brute(18))   # False

**Optimal:** Stop searching at √n instead of n, and — as a small further refinement — handle `2` as a special case up front so the loop can skip every even number afterward (a prime > 2 can never be even, so testing even divisors past 2 is wasted work). This takes it from O(n) to O(√n).

In [ ]:
import math

# Optimal — search only up to sqrt(n), skip even candidates after 2.
# O(sqrt(n)) time.

def is_prime_optimal(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, math.isqrt(n) + 1, 2):   # odd candidates only, isqrt already returns int
        if n % i == 0:
            return False
    return True

print(is_prime_optimal(17))   # True
print(is_prime_optimal(18))   # False

In [ ]:
# Tests

assert is_prime_optimal(2) is True            # edge: smallest prime
assert is_prime_optimal(3) is True
assert is_prime_optimal(1) is False           # edge: 1 is NOT prime, by convention
assert is_prime_optimal(0) is False           # edge
assert is_prime_optimal(-7) is False          # edge: negative numbers aren't prime
assert is_prime_optimal(4) is False           # smallest composite
assert is_prime_optimal(9) is False           # perfect square — must be caught by the sqrt boundary
assert is_prime_optimal(97) is True           # larger prime
assert is_prime_optimal(100) is False

for n in range(-5, 200):
    assert is_prime_brute(n) == is_prime_optimal(n), n

print("All Q6 tests passed.")

**Interview traps:**
- **`1` is not prime.** This trips up more candidates than any other part of this question — it feels like it should be prime ("only divisible by 1 and itself" is technically true of 1!), but it's excluded by definition specifically so that the Fundamental Theorem of Arithmetic (every integer has a *unique* prime factorization) holds. State it as a known convention, not a guess.
- **The sqrt boundary is inclusive.** For a perfect square like `n = 9`, the divisor `3` equals `math.isqrt(9)` exactly — using `range(2, sqrt(n))` (exclusive) instead of `range(2, sqrt(n) + 1)` silently misses that case and misclassifies every perfect square of a prime as prime.
- **`n ** 0.5` vs `math.isqrt(n)`** — floating point square roots can be off by a tiny epsilon for large `n`, which can shift the boundary by one and cause rare, hard-to-reproduce bugs. `math.isqrt` (integer square root, exact) sidesteps this entirely; prefer it once you're past the "explain the idea" stage.
- **Don't forget to reject `n < 2` first.** Falling into the loop for `n = 0` or `n = 1` with `range(2, ...)` happens to return `True` (empty loop, "no divisors found") unless you explicitly guard against it — a classic case of the code *looking* like it handles the edge case by accident, when it actually doesn't.

---
## Q7. List Primes up to N

> **List Primes up to N:** List all prime numbers up to a given number N.
> _Hint:_ Try a brute force prime check for each number or use the Sieve of Eratosthenes for efficiency.

**Restate:** Given a number `N`, produce the list of every prime number that is ≤ `N`.

**Algorithmic thinking (no syntax), brute version:** For every number from `2` up to `N`, run the "is this prime" test from Q6, and keep the ones that pass.

**Algorithmic thinking (no syntax), the better idea:** Instead of asking "is each number prime?" one at a time, flip the question around: start by *assuming* every number from `2` to `N` is a candidate prime. Then, for each number you confirm is prime, go through and cross out every multiple of it (it can't be prime — it's divisible by something other than 1 and itself). Whatever is never crossed out by the time you're done is prime. This is the Sieve of Eratosthenes — you're eliminating composites in bulk instead of testing each number in isolation.

In [ ]:
# Brute force — reuse the Q6 primality test on every candidate.
# Roughly O(N * sqrt(N)) time using is_prime_optimal (or O(N^2) with
# is_prime_brute) — correct, but re-derives primality from scratch for
# every single number, with no memory of what earlier numbers taught us.

def primes_up_to_brute(N):
    return [n for n in range(2, N + 1) if is_prime_optimal(n)]

print(primes_up_to_brute(30))   # [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

**Optimal:** The Sieve of Eratosthenes, O(N log log N) time — for practical purposes this is "almost linear." Two details make it correct and fast rather than just fast:

1. **Start crossing out multiples of `p` at `p × p`, not `2 × p`.** Every smaller multiple of `p` (like `2p`, `3p`, ..., `(p-1)p`) has already been crossed out by a *smaller* prime factor by the time you get to `p` — re-crossing them is wasted work.
2. **Stop advancing `p` once `p × p > N`.** Any composite ≤ N must have a prime factor ≤ √N (same pairing argument as Q6), so once `p` exceeds √N there's nothing left to sieve.

In [ ]:
# Optimal — Sieve of Eratosthenes. O(N log log N) time, O(N) space.

def primes_up_to_sieve(N):
    if N < 2:
        return []
    is_candidate = [True] * (N + 1)   # index i represents the number i
    is_candidate[0] = is_candidate[1] = False   # 0 and 1 are never prime
    for p in range(2, math.isqrt(N) + 1):
        if is_candidate[p]:
            for multiple in range(p * p, N + 1, p):
                is_candidate[multiple] = False
    return [n for n, candidate in enumerate(is_candidate) if candidate]

print(primes_up_to_sieve(30))   # [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

In [ ]:
# Tests

assert primes_up_to_sieve(1) == []             # edge: N below the first prime
assert primes_up_to_sieve(0) == []             # edge
assert primes_up_to_sieve(-5) == []            # edge: negative N
assert primes_up_to_sieve(2) == [2]            # edge: N is itself prime, and the boundary
assert primes_up_to_sieve(3) == [2, 3]
assert primes_up_to_sieve(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert primes_up_to_sieve(4) == [2, 3]         # edge: N is a perfect square, N=4=2*2

for N in range(0, 100):
    assert primes_up_to_sieve(N) == primes_up_to_brute(N), N

print("All Q7 tests passed.")

**Interview traps:**
- **"Up to N" is inclusive.** `range(2, N + 1)` and `[True] * (N + 1)` both need that `+ 1` — a very easy off-by-one to drop, and it silently produces a list missing `N` itself whenever `N` happens to be prime (as in the `N=2` test above).
- **Don't restart the inner loop at `2 * p`.** It's not *wrong* to do so — you still get the correct final list — but it re-marks numbers that were already marked, which is the exact wasted work the sieve exists to avoid. If an interviewer asks "why `p * p`?", the pairing argument from Q6 is the answer they want.
- **The sieve trades space for time.** It needs O(N) memory for the boolean array — for very large N where memory is the bottleneck (not time), the brute-force per-number check with O(1) space (Q6's `is_prime_optimal`) can actually be the right trade-off. Know both, and know *why* you'd pick one over the other, rather than assuming "sieve is always better."
- **List comprehension building the final list is O(N)**, on top of the O(N log log N) sieving — that's fine and expected, don't mistake it for a second sieve pass that needs its own optimization.

---
## Q8. Greatest Common Divisor

> **Greatest Common Divisor:** Find the greatest common divisor (GCD) of two positive integers.
> _Hint:_ Use Euclid's algorithm (repeatedly take modulo) for an efficient solution.

**Restate:** Given two positive integers `a` and `b`, find the largest whole number that divides both of them with no remainder.

**Algorithmic thinking (no syntax), brute version:** Try every whole number starting from the smaller of the two, counting downward. The first one that divides *both* numbers evenly is the greatest common divisor (since you're checking from largest candidate to smallest, the first hit is automatically the biggest one).

**Algorithmic thinking (no syntax), the better idea:** Notice that the GCD of two numbers doesn't change if you replace the larger one with the *remainder* left over after dividing it by the smaller one. (Any number that divides both `a` and `b` also divides `a mod b`, and vice versa — they share exactly the same set of common divisors.) So: replace the larger number with that remainder, and repeat with the new, smaller pair. Keep shrinking the pair this way until one of them hits zero — whatever's left in the other slot is the GCD. This is Euclid's algorithm, and it shrinks the numbers *fast* (roughly by half every two steps), not one-at-a-time like the brute force.

In [ ]:
# Brute force — count down from min(a, b), first common divisor wins.
# O(min(a, b)) time in the worst case (e.g. two coprime numbers).

def gcd_brute(a, b):
    for candidate in range(min(a, b), 0, -1):
        if a % candidate == 0 and b % candidate == 0:
            return candidate
    return 1   # unreachable in practice: 1 always divides both

print(gcd_brute(48, 18))   # 6
print(gcd_brute(17, 5))    # 1 — coprime

**Optimal:** Euclid's algorithm — O(log(min(a, b))) time, which is a dramatic jump. For 30-digit numbers, the brute force is computationally impossible; Euclid's algorithm still finishes in well under 100 steps.

In [ ]:
# Optimal — Euclid's algorithm, iterative. O(log(min(a, b))) time.

def gcd_optimal(a, b):
    while b != 0:
        a, b = b, a % b   # "swap in" the remainder, shrinking the pair each step
    return a

print(gcd_optimal(48, 18))   # 6
print(gcd_optimal(17, 5))    # 1
print(gcd_optimal(48, 18) == math.gcd(48, 18))   # sanity check against the stdlib

In [ ]:
# Tests

assert gcd_optimal(48, 18) == 6
assert gcd_optimal(17, 5) == 1                # edge: coprime numbers
assert gcd_optimal(7, 7) == 7                 # edge: equal numbers
assert gcd_optimal(1, 100) == 1               # edge: one operand is 1
assert gcd_optimal(100, 1) == 1               # order shouldn't matter
assert gcd_optimal(0, 5) == 5                 # edge: gcd(0, n) is n, by definition
assert gcd_optimal(5, 0) == 5
assert gcd_optimal(48, 18) == gcd_optimal(18, 48)   # symmetric regardless of argument order

for a in range(1, 60):
    for b in range(1, 60):
        assert gcd_optimal(a, b) == gcd_brute(a, b) == math.gcd(a, b), (a, b)

print("All Q8 tests passed.")

**Interview traps:**
- **Don't check `a % b == 0` and stop.** A common wrong shortcut is "if the bigger one is divisible by the smaller one, the smaller one is the GCD, otherwise give up" — that only handles the case where one number already divides the other. Euclid's algorithm's repeated step is what handles the general case.
- **The spec says "positive integers,"** but interviewers often probe `gcd(0, n)` anyway. Mathematically `gcd(0, n) = n` (every number divides 0, so the *greatest* common divisor of `0` and `n` is just `n`). The iterative Euclid implementation above handles this for free — trace through it (`a=0, b=5` → swap to `a=5, b=0` → loop ends → return `5`) to confirm you understand why, rather than asserting it works because it "just does."
- **`gcd(0, 0)` is undefined** (every number divides both, so there's no single greatest one) — `math.gcd(0, 0)` returns `0` by convention. Worth naming as an edge case you'd clarify rather than silently returning something and hoping.
- **Recursive Euclid is equally valid** (`return a if b == 0 else gcd(b, a % b)`) and arguably reads closer to the "definition." It's fine to offer either — the recursion depth here is O(log(min(a,b))), nowhere near deep enough to hit Python's recursion limit, unlike the naive Fibonacci recursion in Q5.
- **In real code, use `math.gcd`** — reimplementing this outside an interview is pure risk for no benefit, exactly like `math.factorial` in Q4.

---
## Q9. Count Vowels

> **Count Vowels:** Count the number of vowels in a given string.
> _Hint:_ Loop through the string or use a list comprehension, checking each character against vowels `{'a','e','i','o','u'}`.

**Restate:** Given a string, count how many of its characters are one of `a, e, i, o, u` (we'll decide how to handle uppercase explicitly, since the question doesn't say).

**Algorithmic thinking (no syntax):**
1. Start a count at zero.
2. Look at each character in the string, one at a time.
3. If that character is one of the five vowels, add one to the count.
4. After looking at every character, the count is the answer.

Unlike most of the questions so far, there's no hidden "repeated work" to eliminate here — every character has to be looked at exactly once no matter what, so both versions below are O(n) time. What actually changes between "brute" and "optimal" in this case is the *cost of the membership check itself*, and how idiomatically it's expressed — worth knowing, but don't expect a Big-O win.

In [ ]:
# Brute force — explicit loop, membership check against a *list*.
# Still O(n) overall, but each "in" check against a list is itself
# O(k) where k = number of vowels (constant here, so it doesn't
# change the asymptotics — but it's the wrong habit to build, since
# k won't always stay constant and small in other problems).

def count_vowels_brute(s):
    vowels = ['a', 'e', 'i', 'o', 'u']
    count = 0
    for char in s:
        if char in vowels:
            count += 1
    return count

print(count_vowels_brute("hello world"))   # 3

**Optimal:** Use a `set` instead of a `list` for the membership check — O(1) average per lookup instead of O(k) — and handle case explicitly by lowercasing each character before checking, since the spec is ambiguous and real-world sentences have capitals. Expressed as a generator passed to `sum()`, which reads as "the count of characters satisfying this condition" rather than a manual accumulator.

In [ ]:
# Optimal — set membership, case-insensitive, generator + sum().

VOWELS = frozenset('aeiou')

def count_vowels_optimal(s):
    return sum(1 for char in s.lower() if char in VOWELS)

print(count_vowels_optimal("hello world"))   # 3
print(count_vowels_optimal("HELLO WORLD"))   # 3 — case handled

In [ ]:
# Tests

assert count_vowels_optimal("") == 0                       # edge: empty string
assert count_vowels_optimal("bcdfg") == 0                  # edge: no vowels at all
assert count_vowels_optimal("aeiou") == 5                  # edge: all vowels
assert count_vowels_optimal("AEIOU") == 5                  # edge: all vowels, uppercase
assert count_vowels_optimal("hello world") == 3
assert count_vowels_optimal("Hello, World!") == 3          # punctuation ignored automatically
assert count_vowels_optimal("y") == 0                      # 'y' is NOT counted as a vowel here

assert count_vowels_brute("hello world") == count_vowels_optimal("hello world".lower())

print("All Q9 tests passed.")

**Interview traps:**
- **Case sensitivity is almost never specified but always tested.** The question's own hint set — `{'a','e','i','o','u'}` — is lowercase-only. If you check `char in vowels` on raw input without lowercasing first, `"HELLO"` silently counts as zero vowels. State your assumption ("I'll treat this case-insensitively") before coding, then handle it.
- **`'y'` is a genuinely ambiguous vowel in English** ("sky" vs "yellow") but essentially every interview version of this question treats it as *not* a vowel unless explicitly told otherwise. Don't add it on your own initiative.
- **`in` against a `list` vs a `set`** looks identical in code but isn't the same operation — this is a good moment to demonstrate you know Python's data structures have different Big-O for membership testing (list: O(k) linear scan; set: O(1) average via hashing) even when it doesn't move the needle for a 5-element vowel set. The same habit matters a lot more in Q20/Q21 (list intersection / subset checks) later in this set.
- **Don't forget non-letter characters.** Digits, punctuation, and whitespace should just fail the membership check and contribute nothing — if you instead try to validate that every character is a letter first, you've added unnecessary and unrequested complexity.

---
## Q10. Word Frequency

> **Word Frequency:** Given a sentence, count the occurrences of each word.
> _Hint:_ Use `str.split()` to get words and a dictionary (or `collections.Counter`) to count frequencies.

**Restate:** Given a sentence, produce a mapping from each distinct word to how many times it appears.

**Algorithmic thinking (no syntax):**
1. Break the sentence apart into individual words.
2. Keep a running tally, one entry per distinct word seen so far, starting each new word's tally at zero.
3. Go through the words in order; for each one, add one to its tally.
4. Once every word has been processed, the tally is the answer.

The interesting design decision here isn't algorithmic complexity (both versions below are O(n) in the number of words) — it's **what counts as "the same word."** "The" and "the" — same word or not? "cat," and "cat" — does trailing punctuation count? The brute-force version below makes the naive choice (whatever `.split()` gives you, verbatim); the optimal version makes that choice deliberately and states it.

In [ ]:
# Brute force — manual dict, verbatim words (case- and punctuation-sensitive).
# O(n) time in the number of words, O(k) space for k distinct words.

def word_frequency_brute(sentence):
    counts = {}
    for word in sentence.split():
        if word in counts:
            counts[word] += 1
        else:
            counts[word] = 1
    return counts

print(word_frequency_brute("the cat sat on the mat the cat ran"))
# {'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1, 'ran': 1}

**Optimal:** Same O(n) time, but two improvements:

- **`collections.Counter`** removes the "does this key exist yet?" branch entirely — it's a `dict` subclass where missing keys default to `0`, so `counts[word] += 1` always just works. It also comes with `.most_common(k)` for free, which is almost always the next thing this question's follow-up asks for.
- **Normalize the words deliberately** — lowercase them, and strip surrounding punctuation — so `"Cat"`, `"cat,"`, and `"cat"` are correctly treated as the same word instead of three different dictionary keys.

In [ ]:
import string
from collections import Counter

# Optimal — Counter + explicit normalization (lowercase, strip punctuation).

def word_frequency_optimal(sentence):
    words = [
        word.strip(string.punctuation).lower()
        for word in sentence.split()
    ]
    words = [word for word in words if word]   # drop anything that was pure punctuation
    return Counter(words)

result = word_frequency_optimal("The cat sat on the mat. The cat ran!")
print(result)                        # Counter({'the': 3, 'cat': 2, ...})
print(result.most_common(2))         # [('the', 3), ('cat', 2)]

In [ ]:
# Tests

assert word_frequency_optimal("") == Counter()                        # edge: empty sentence
assert word_frequency_optimal("hello") == Counter({"hello": 1})       # edge: single word
assert word_frequency_optimal("cat cat cat") == Counter({"cat": 3})   # all repeats
assert word_frequency_optimal("Cat cat CAT") == Counter({"cat": 3})   # case normalized
assert word_frequency_optimal("cat, cat. cat!") == Counter({"cat": 3})  # punctuation stripped
assert word_frequency_optimal("a  b   c") == Counter({"a": 1, "b": 1, "c": 1})  # multiple spaces

# .split() with no argument collapses runs of whitespace and drops leading/
# trailing whitespace automatically -- unlike .split(" "), which would
# produce empty-string "words" for every extra space.
assert "  a  b  ".split() == ["a", "b"]

assert word_frequency_brute("the cat sat") == {"the": 1, "cat": 1, "sat": 1}

print("All Q10 tests passed.")

**Interview traps:**
- **`.split()` vs `.split(" ")` are not the same method call.** `.split()` (no argument) treats any run of whitespace as one separator and ignores leading/trailing whitespace. `.split(" ")` splits on *literal single spaces*, so `"a  b".split(" ")` gives `["a", "", "b"]` — an empty-string "word" for the double space. This is a real, silent-bug-producing trap, not a style nitpick.
- **Case and punctuation normalization is a judgment call you should name out loud**, exactly like the case-sensitivity call in Q9. "The" and "the" being counted separately (or not), and "cat," and "cat" being counted separately (or not), are both defensible depending on what the frequency count is *for* — don't silently pick one.
- **`dict.get(word, 0) + 1` is a clean middle ground** between the manual `if/else` and `Counter` if you want to show you know the pattern without reaching for an import: `counts[word] = counts.get(word, 0) + 1`. Worth having in your pocket.
- **`Counter` is a subclass of `dict`**, so everything you know about dicts still applies (iteration order is insertion order, `in` checks work the same way) — it isn't a mysterious separate type, just a `dict` with a friendlier default and `.most_common()`.
- **Mutating a dict while iterating over it** raises `RuntimeError` — not directly triggered by this question, but this is the classic trap that *does* bite people who reach for a dict-based counting pattern in a slightly different shape (e.g., trying to remove entries with count 1 in the same loop that builds the counts).